In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


In [ ]:
pip install transformers

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
for metin in ["gelcekmsn", "halısaya aldık", "b3dava", "Değerli müşterimiz kartınız bloke oldu [LINK] tıklayın"]:
    print(tok.tokenize(metin))
    

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)
model = model.to("cuda")

print(sum(p.numel() for p in model.parameters()) / 1e6, "milyon parametre")
print(torch.cuda.memory_allocated() / 1e9, "GB GPU belleği kullanılıyor")

In [ ]:
import sys
sys.path.append("..")
import pandas as pd

train = pd.read_csv("../data/processed/tr_train.csv")
train = train[train["belirsiz"] != "e"]

uzunluklar = train["mesaj"].apply(lambda m: len(tok(m)["input_ids"]))
print(uzunluklar.describe())
print("95. yüzdelik:", uzunluklar.quantile(0.95))




In [ ]:
import time
import numpy as np
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

del model
torch.cuda.empty_cache()

AYAR = {"model": "FacebookAI/xlm-roberta-base", "max_len": 80, "batch": 16,
        "lr": 2e-5, "epoch": 4, "warmup": 0.1, "weight_decay": 0.01, "seed": 42}

X = train["mesaj"].tolist()
y = (train["ikili"] == "spam").astype(int).to_numpy()
katlar = train["fold"].to_numpy()

def yukleyici(metinler, etiketler, a, karistir):
    veri = list(zip(metinler, etiketler))
    def birlestir(parti):
        m, e = zip(*parti)
        kod = tok(list(m), padding=True, truncation=True, max_length=a["max_len"], return_tensors="pt")
        kod["labels"] = torch.tensor(e)
        return kod
    return DataLoader(veri, batch_size=a["batch"], shuffle=karistir, collate_fn=birlestir)

def egit_ve_tahmin(X_e, y_e, X_d, a):
    torch.manual_seed(a["seed"])
    m = AutoModelForSequenceClassification.from_pretrained(a["model"], num_labels=2).to("cuda")
    egitim = yukleyici(X_e, y_e, a, karistir=True)
    opt = torch.optim.AdamW(m.parameters(), lr=a["lr"], weight_decay=a["weight_decay"])
    toplam = len(egitim) * a["epoch"]
    plan = get_linear_schedule_with_warmup(opt, int(toplam * a["warmup"]), toplam)

    m.train()
    for ep in range(a["epoch"]):
        for parti in egitim:
            parti = {k: v.to("cuda") for k, v in parti.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                kayip = m(**parti).loss
            kayip.backward()
            opt.step()
            plan.step()
            opt.zero_grad()

    m.eval()
    olasilik = []
    with torch.no_grad():
        for parti in yukleyici(X_d, [0] * len(X_d), a, karistir=False):
            parti = {k: v.to("cuda") for k, v in parti.items() if k != "labels"}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logit = m(**parti).logits
            olasilik.append(torch.softmax(logit.float(), dim=-1)[:, 1].cpu().numpy())

    del m, opt
    torch.cuda.empty_cache()
    return np.concatenate(olasilik)

basla = time.time()
egit, dogr = katlar != 0, katlar == 0
p = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit], [X[i] for i in np.where(dogr)[0]], AYAR)
print(f"fold 0 F1: {f1_score(y[dogr], (p > 0.5).astype(int)):.3f}")
print(f"süre: {time.time() - basla:.0f} sn, en yüksek GPU belleği: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
import mlflow
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("tr_ikili")

def xlmr_cv(a):
    skorlar = []
    for k in range(5):
        egit, dogr = katlar != k, katlar == k
        p = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit],
                           [X[i] for i in np.where(dogr)[0]], a)
        skorlar.append(f1_score(y[dogr], (p > 0.5).astype(int)))
        print(f"fold {k}: {skorlar[-1]:.3f}")
    return np.array(skorlar)

with mlflow.start_run(run_name="xlmr_base"):
    skor = xlmr_cv(AYAR)
    mlflow.log_params(AYAR)
    mlflow.log_metrics({"cv_f1": skor.mean(), "cv_f1_std": skor.std()})
    print(skor.mean().round(3), skor.round(3))

In [ ]:
oof = np.zeros(len(y))
for k in range(5):
    egit, dogr = katlar != k, katlar == k
    oof[dogr] = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit],
                               [X[i] for i in np.where(dogr)[0]], AYAR)

train["xlmr_olasilik"] = oof
hatalar = train[(oof > 0.5).astype(int) != y]
print(len(hatalar), "hata,", (hatalar["fold"] == 2).sum(), "tanesi fold 2'de")
hatalar[["id", "fold", "ikili", "xlmr_olasilik", "mesaj"]].sort_values("fold")

In [ ]:
zor = pd.read_csv("C:\\Users\\SCT\\Projects\\sms-fraud-detection\\external\\zor_gercek_hepsi.csv")
zor = zor = zor[zor["etiket"].notna()].reset_index(drop=True)
zor["y"] = (zor["etiket"] != "meşru").astype(int)

print(zor.shape, zor["y"].value_counts().to_dict(), zor["gonderen"].nunique(), "gönderici")
uzun = zor["mesaj"].apply(lambda m: len(tok(m)["input_ids"]))
print("80'i aşan:", (uzun > 80).sum(), " en uzun:", uzun.max())

In [ ]:
AYAR_Z = {**AYAR, "max_len": 160}

def zor_olc(y_gercek, olasilik):
    t = (olasilik > 0.5).astype(int)
    meşru, spam = y_gercek == 0, y_gercek == 1
    return {
        "zor_f1": f1_score(y_gercek, t),
        "zor_p": precision_score(y_gercek, t),
        "zor_r": recall_score(y_gercek, t),
        "yanlis_alarm": int(t[meşru].sum()),
        "spam_yakalama": int(t[spam].sum()),
    }

from sklearn.metrics import precision_score, recall_score

basla = time.time()
p_A = egit_ve_tahmin(X, y, zor["mesaj"].tolist(), AYAR_Z)
sonuc_A = zor_olc(zor["y"].to_numpy(), p_A)

with mlflow.start_run(run_name="xlmr_zor_A"):
    mlflow.log_params({**AYAR_Z, "egitim": "tr_train", "sinav": "zor_gercek_133"})
    mlflow.log_metrics(sonuc_A)

print(sonuc_A)
print(f"yanlış alarm {sonuc_A['yanlis_alarm']}/68, spam yakalama {sonuc_A['spam_yakalama']}/65, "
      f"süre {time.time() - basla:.0f} sn, GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
zor["grup"] = zor["gonderen"].str.upper().str.strip()
gruplar = zor["grup"].unique()
print(len(gruplar), "gönderici grubu")

p_B = np.zeros(len(zor))
basla = time.time()
for i, g in enumerate(gruplar):
    sinav = (zor["grup"] == g).to_numpy()
    ek = zor[~sinav]
    X_e = X + ek["mesaj"].tolist()
    y_e = np.concatenate([y, ek["y"].to_numpy()])
    p_B[sinav] = egit_ve_tahmin(X_e, y_e, zor.loc[sinav, "mesaj"].tolist(), AYAR_Z)
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(gruplar)} bitti, {(time.time() - basla) / 60:.1f} dk")

sonuc_B = zor_olc(zor["y"].to_numpy(), p_B)
with mlflow.start_run(run_name="xlmr_zor_B"):
    mlflow.log_params({**AYAR_Z, "egitim": "tr_train + diğer göndericiler", "sinav": "leave-one-sender-out (büyük harf)"})
    mlflow.log_metrics(sonuc_B)

zor["p_A"], zor["p_B"] = p_A, p_B
print(sonuc_B)
print(f"yanlış alarm {sonuc_B['yanlis_alarm']}/68, spam yakalama {sonuc_B['spam_yakalama']}/65, "
      f"GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
# --- 1. Tahminleri etikete çevir ---
zor["t_A"] = (zor["p_A"] > 0.5).astype(int)
zor["t_B"] = (zor["p_B"] > 0.5).astype(int)

# --- 2. Hangi mesajlar A'da yanlış, B'de doğru oldu? (düzelenler) ---
duzelen = zor[(zor["t_A"] != zor["y"]) & (zor["t_B"] == zor["y"])]
print("Düzelen:", len(duzelen))

# --- 3. B'de hâlâ yanlış olanlar ---
kalan = zor[zor["t_B"] != zor["y"]].copy()
kalan["hata"] = np.where(kalan["y"] == 0, "yanlış alarm", "kaçan spam")   # hata türü etiketi
print("B'de hâlâ yanlış:", len(kalan))

# --- 4. Dolandırıcılık özelinde: 11 gerçek dolandırıcılığın kaçı yakalandı? ---
dol = zor["etiket"] == "dolandırıcılık"
print(f"Dolandırıcılık yakalama  A: {zor.loc[dol, 't_A'].sum()}/{dol.sum()}  "
      f"B: {zor.loc[dol, 't_B'].sum()}/{dol.sum()}")

# --- 5. Kalan hataları göster (kısa mesaj önizlemesiyle) ---
kalan["onizleme"] = kalan["mesaj"].str[:70]          # ilk 70 karakter, tablo taşmasın
kalan.sort_values("hata")[["hata", "grup", "etiket", "p_A", "p_B", "onizleme"]].round(2)

In [ ]:
# --- 1. Paylaşılacak sütunları seç ---
cikti = zor[["gonderen", "grup", "etiket", "y", "p_A", "p_B", "t_A", "t_B", "mesaj"]]

# --- 2. Gitignore'daki klasöre yaz (repoya girmez) ---
cikti.to_csv("C:\\Users\\SCT\\Projects\\sms-fraud-detection\\external\\zor_gercek_hepsi.csv", index=False, encoding="utf-8-sig")

# --- 3. Dolandırıcılık özetini tekrar yazdır (önceki çıktıda görünmemişti) ---
dol = zor["etiket"] == "dolandırıcılık"
print(f"Dolandırıcılık yakalama  A: {zor.loc[dol, 't_A'].sum()}/{dol.sum()}  "
      f"B: {zor.loc[dol, 't_B'].sum()}/{dol.sum()}")

In [ ]:
# --- Zor seti oku (göreli yol: notebooks/ klasöründen bir üst klasöre çık) ---
zor = pd.read_csv("../data/external/zor_gercek_hepsi.csv")